# LeetCode 1760 — Minimum Limit of Balls in a Bag bằng Python

Link đề: https://leetcode.com/problems/minimum-limit-of-balls-in-a-bag/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def minimumSize(self, nums: List[int], maxOperations: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Đổi bài "tối ưu" sang bài "**kiểm tra khả thi**": với mức phạt `p` cho trước, cần ít nhất bao nhiêu thao tác?
2. Rút ra công thức số thao tác cho một túi: `⌈x/p⌉ − 1 = (x − 1) // p`.
3. Thấy tính **đơn điệu** (`p` càng lớn càng dễ) ⇒ **tìm kiếm nhị phân trên đáp án** (binary search on answer).
4. Hiểu vì sao tham lam "luôn chia đôi túi lớn nhất" **sai**.
5. Kiểm chứng bằng vét cạn mọi cách chia trên input rất nhỏ.

## 2. Đề bài & ví dụ

Cho mảng `nums`, túi thứ `i` có `nums[i]` quả bóng, và số nguyên `maxOperations`. Được làm **tối đa** `maxOperations` lần thao tác: chọn một túi và **chia** nó thành hai túi mới, mỗi túi có số bóng **dương** (ví dụ túi 5 → `1 + 4` hoặc `2 + 3`).

**Mức phạt** là số bóng **lớn nhất** trong một túi. Hãy làm mức phạt **nhỏ nhất** có thể và trả về giá trị đó.

**Ví dụ 1:** `nums = [9], maxOperations = 2` → `3`: `[9] → [6,3] → [3,3,3]`.

**Ví dụ 2:** `nums = [2,4,8,2], maxOperations = 4` → `2`: chia `8 → 4+4`, rồi ba túi `4` thành `2+2`.

**Ràng buộc:** `1 ≤ nums.length ≤ 10⁵`, `1 ≤ maxOperations, nums[i] ≤ 10⁹`.

In [1]:
import random
import time
import bisect
import heapq
from functools import lru_cache
from typing import List

examples = [([9], 2, 3), ([2, 4, 8, 2], 4, 2)]
for nums, ops, exp in examples:
    print(f"nums={nums}, maxOperations={ops} -> kỳ vọng {exp}")

nums=[9], maxOperations=2 -> kỳ vọng 3
nums=[2, 4, 8, 2], maxOperations=4 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1 Chia lấy trần `⌈a/b⌉` bằng số nguyên

`⌈a/b⌉ = (a + b − 1) // b = −(−a // b)` với `a, b > 0`. Tránh `math.ceil(a / b)` vì số thực có thể sai với số lớn.

In [2]:
for a, b in [(9, 3), (10, 3), (1, 5), (10 ** 18 + 1, 2)]:
    print(a, b, "->", (a + b - 1) // b, -(-a // b))

9 3 -> 3 3
10 3 -> 4 4
1 5 -> 1 1
1000000000000000001 2 -> 500000000000000001 500000000000000001


### 3.2 Tìm kiếm nhị phân "giá trị nhỏ nhất thỏa điều kiện"

Nếu `ok(p)` **đơn điệu tăng** (`False … False True … True`), tìm `p` nhỏ nhất có `ok(p) = True`:

```python
lo, hi = L, R          # ok(R) chắc chắn đúng
while lo < hi:
    mid = (lo + hi) // 2
    if ok(mid): hi = mid
    else:       lo = mid + 1
return lo
```

In [3]:
def first_true(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid): hi = mid
        else: lo = mid + 1
    return lo

print(first_true(1, 100, lambda x: x * x >= 50), first_true(1, 10, lambda x: True), first_true(1, 10, lambda x: x == 10))

8 1 10


### 3.3 `bisect` với `key` (Python ≥ 3.10)

`bisect_left(range(1, M + 1), True, key=ok)` trả chỉ số đầu tiên có `ok = True` (vì `False < True`), cộng 1 ra giá trị.

In [4]:
idx = bisect.bisect_left(range(1, 101), True, key=lambda x: x * x >= 50)
print("chỉ số:", idx, "-> giá trị:", idx + 1)

chỉ số: 7 -> giá trị: 8


### 3.4 `heapq` với số âm (heap lớn nhất)

Dùng để cài tham lam "chia túi lớn nhất" (sẽ thấy là sai ở mục 13).

In [5]:
h = [-x for x in [2, 4, 8, 2]]; heapq.heapify(h)
print("lớn nhất:", -h[0])

lớn nhất: 8


## 4. Ý tưởng nền tảng / trực giác

Đặt câu hỏi ngược: **"Nếu muốn mức phạt ≤ p thì cần bao nhiêu thao tác?"**

Túi `x` phải thành các túi `≤ p` ⇒ cần ít nhất `⌈x/p⌉` túi con ⇒ `⌈x/p⌉ − 1` lần chia (mỗi lần chia tăng số túi thêm 1). Và **đủ**: cứ tách ra từng túi `p` một.

Tính tay **Ví dụ 1** (`x = 9`):

| p | ⌈9/p⌉ − 1 | ≤ 2? |
|---|---|---|
| 1 | 8 | không |
| 2 | 4 | không |
| 3 | 2 | **có** |
| 4 | 2 | có |
| 5 | 1 | có |

⇒ `p` nhỏ nhất khả thi là 3. Bảng cột phải có dạng `không … không có … có` — **đơn điệu**.

In [6]:
def ops_needed(nums, p):
    return sum((x - 1) // p for x in nums)

for nums, ops, exp in examples:
    print(nums, ops, [(p, ops_needed(nums, p), ops_needed(nums, p) <= ops) for p in range(1, max(nums) + 1)])

[9] 2 [(1, 8, False), (2, 4, False), (3, 2, True), (4, 2, True), (5, 1, True), (6, 1, True), (7, 1, True), (8, 1, True), (9, 0, True)]
[2, 4, 8, 2] 4 [(1, 12, False), (2, 4, True), (3, 3, True), (4, 1, True), (5, 1, True), (6, 1, True), (7, 1, True), (8, 0, True)]


## 5. Cách 1 — Brute Force / baseline: vét cạn mọi cách chia (chỉ cho input rất nhỏ)

**Ý tưởng:** trạng thái = multiset các túi + số thao tác còn lại. Thử chia mọi túi `x ≥ 2` thành `(a, x − a)` với `1 ≤ a ≤ x/2`, hoặc dừng. Chuẩn hóa bằng `tuple(sorted(...))` và ghi nhớ. Đây là định nghĩa thuần túy — dùng làm đáp án chuẩn.

In [7]:
def penalty_bruteforce(nums, maxOperations):
    @lru_cache(maxsize=None)
    def best(bags, k):
        res = max(bags)
        if k == 0:
            return res
        for idx, x in enumerate(bags):
            if x >= 2 and (idx == 0 or bags[idx - 1] != x):         # tránh thử trùng túi bằng nhau
                rest = bags[:idx] + bags[idx + 1:]
                for a in range(1, x // 2 + 1):
                    res = min(res, best(tuple(sorted(rest + (a, x - a))), k - 1))
        return res
    return best(tuple(sorted(nums)), maxOperations)

for nums, ops, exp in examples:
    got = penalty_bruteforce(nums, ops); print(got, "kỳ vọng", exp); assert got == exp

3 kỳ vọng 3
2 kỳ vọng 2


**Độ phức tạp:** bùng nổ tổ hợp — chỉ chạy được khi tổng bóng và số thao tác rất nhỏ.

### Cách 1' — Thử mọi mức phạt `p` tăng dần

Dùng công thức ở mục 4: `p` đầu tiên có `ops_needed(p) ≤ maxOperations` là đáp án.

In [8]:
def penalty_linear(nums, maxOperations):
    for p in range(1, max(nums) + 1):
        if sum((x - 1) // p for x in nums) <= maxOperations:
            return p

for nums, ops, exp in examples:
    assert penalty_linear(nums, ops) == exp
print([penalty_linear(n, o) for n, o, _ in examples])

[3, 2]


**Độ phức tạp Cách 1':** `O(max(nums) · n)` — tệ nhất `10⁹ · 10⁵` ⇒ **TLE**.

**Nút thắt:** thử `p` lần lượt từng giá trị trong khi tính chất "khả thi" là **đơn điệu**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (kiểm tra khả thi):** `ok(p) ⇔ Σ (x − 1) // p ≤ maxOperations`.

**Insight 2 (đơn điệu):** nếu `p` khả thi thì mọi `p' > p` cũng khả thi (`(x − 1) // p` không tăng khi `p` tăng) ⇒ tìm `p` nhỏ nhất bằng **nhị phân** trên `[1, max(nums)]` (`p = max(nums)` luôn khả thi với 0 thao tác).

**Insight 3 (cận dưới tốt hơn):** sau `maxOperations` lần chia có tối đa `n + maxOperations` túi ⇒ mức phạt `≥ ⌈Σnums / (n + maxOperations)⌉`. Thu hẹp `lo` giúp giảm số vòng (không đổi độ phức tạp).

**Quy tắc:** nếu `ok(mid)` thì đáp án `≤ mid` (`hi = mid`); ngược lại đáp án `> mid` (`lo = mid + 1`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tìm kiếm nhị phân trên đáp án

In [9]:
def penalty_binary(nums, maxOperations):
    lo, hi = 1, max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((x - 1) // mid for x in nums) <= maxOperations:
            hi = mid
        else:
            lo = mid + 1
    return lo

print([penalty_binary(n, o) for n, o, _ in examples])

[3, 2]


**Code nộp LeetCode (Cách 2):** xem mục 9. **Độ phức tạp:** `O(n · log M)` với `M = max(nums) ≤ 10⁹` ⇒ ~`30 · 10⁵ = 3·10⁶` phép chia.

### 7.2 Cách 3 — `bisect_left` với `key`

In [10]:
class SolutionBisect:
    def minimumSize(self, nums: List[int], maxOperations: int) -> int:
        def ok(p: int) -> bool:
            return sum((x - 1) // p for x in nums) <= maxOperations
        return bisect.bisect_left(range(1, max(nums) + 1), True, key=ok) + 1

print([SolutionBisect().minimumSize(n, o) for n, o, _ in examples])

[3, 2]


**Độ phức tạp Cách 3:** như Cách 2. `range` không tạo list thật nên không tốn bộ nhớ `O(M)`.

### 7.3 Cách 4 — Nhị phân với cận dưới chặt + dừng sớm khi đếm

In [11]:
class SolutionTight:
    def minimumSize(self, nums: List[int], maxOperations: int) -> int:
        n = len(nums)
        lo = max(1, -(-sum(nums) // (n + maxOperations)))
        hi = max(nums)
        def ok(p: int) -> bool:
            need = 0
            for x in nums:
                need += (x - 1) // p
                if need > maxOperations:          # dừng sớm
                    return False
            return True
        while lo < hi:
            mid = (lo + hi) // 2
            if ok(mid):
                hi = mid
            else:
                lo = mid + 1
        return lo

print([SolutionTight().minimumSize(n, o) for n, o, _ in examples])

[3, 2]


**Vì sao cận dưới đúng:** tổng bóng không đổi, số túi cuối `≤ n + maxOperations`, nên túi lớn nhất `≥` trung bình `≥ ⌈Σ/(n+ops)⌉`. Cận dưới này luôn `≤ max(nums)`.

**Độ phức tạp Cách 4:** `O(n · log(hi − lo))`, thường ít vòng hơn Cách 2.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 2** (`nums = [2,4,8,2]`, `ops = 4`):

| vòng | lo | hi | mid | ops cần | khả thi? |
|---|---|---|---|---|---|
| 1 | 1 | 8 | 4 | 0+0+1+0 = 1 | có → hi=4 |
| 2 | 1 | 4 | 2 | 0+1+3+0 = 4 | có → hi=2 |
| 3 | 1 | 2 | 1 | 1+3+7+1 = 12 | không → lo=2 |
| kết thúc | 2 | 2 | | | **2** |

In [12]:
def trace_binary(nums, maxOperations):
    lo, hi = 1, max(nums); step = 0
    while lo < hi:
        step += 1
        mid = (lo + hi) // 2
        parts = [(x - 1) // mid for x in nums]
        ok = sum(parts) <= maxOperations
        print(f"vòng {step}: lo={lo}, hi={hi}, mid={mid}, thao tác từng túi={parts}, tổng={sum(parts)} -> {'khả thi' if ok else 'không'}")
        if ok: hi = mid
        else: lo = mid + 1
    print("Kết quả:", lo)
    return lo

assert trace_binary([2, 4, 8, 2], 4) == 2
print()
assert trace_binary([9], 2) == 3

vòng 1: lo=1, hi=8, mid=4, thao tác từng túi=[0, 0, 1, 0], tổng=1 -> khả thi
vòng 2: lo=1, hi=4, mid=2, thao tác từng túi=[0, 1, 3, 0], tổng=4 -> khả thi
vòng 3: lo=1, hi=2, mid=1, thao tác từng túi=[1, 3, 7, 1], tổng=12 -> không
Kết quả: 2

vòng 1: lo=1, hi=9, mid=5, thao tác từng túi=[1], tổng=1 -> khả thi
vòng 2: lo=1, hi=5, mid=3, thao tác từng túi=[2], tổng=2 -> khả thi
vòng 3: lo=1, hi=3, mid=2, thao tác từng túi=[4], tổng=4 -> không
Kết quả: 3


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — nhị phân trên đáp án.

In [13]:
class Solution:
    def minimumSize(self, nums: List[int], maxOperations: int) -> int:
        lo, hi = 1, max(nums)
        while lo < hi:
            mid = (lo + hi) // 2
            need = sum((x - 1) // mid for x in nums)
            if need <= maxOperations:
                hi = mid
            else:
                lo = mid + 1
        return lo

for nums, ops, exp in examples:
    got = Solution().minimumSize(nums, ops); print(got, exp); assert got == exp

3 3
2 2


## 10. Vì sao thuật toán đúng

**Bổ đề 1.** Số thao tác tối thiểu để mọi túi con của túi `x` có `≤ p` bóng là `⌈x/p⌉ − 1`.
- *Cần:* sau `k` lần chia, túi `x` thành `k + 1` túi; nếu mỗi túi `≤ p` thì `x ≤ (k+1)p` ⇒ `k ≥ ⌈x/p⌉ − 1`.
- *Đủ:* tách dần các túi `p` bóng ra cho tới khi phần còn lại `≤ p`.

Thao tác trên các túi độc lập nhau ⇒ tổng tối thiểu là `need(p) = Σ (⌈x/p⌉ − 1) = Σ (x − 1) // p`.

**Bổ đề 2 (đơn điệu).** `need(p)` không tăng theo `p` ⇒ `ok(p) = [need(p) ≤ ops]` có dạng `F…F T…T`.

**Nhị phân:** invariant "đáp án thuộc `[lo, hi]`" — ban đầu đúng vì `ok(max) = True` (`need = 0`) và `p ≥ 1`. Nếu `ok(mid)`, đáp án `≤ mid`; nếu không, đáp án `> mid` (Bổ đề 2). Khoảng co lại mỗi vòng, dừng khi `lo = hi` = đáp án. ∎

Đẳng thức `⌈x/p⌉ − 1 = (x − 1) // p`: viết `x − 1 = qp + r`, `0 ≤ r < p` ⇒ `x = qp + r + 1` với `1 ≤ r + 1 ≤ p` ⇒ `⌈x/p⌉ = q + 1`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums) ≤ 10⁵`, `M = max(nums) ≤ 10⁹`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn chia | mũ | mũ |
| 1'. Thử mọi `p` | `O(M · n)` | `O(1)` |
| 2. Nhị phân | `O(n log M)` | `O(1)` |
| 3. `bisect` + key | `O(n log M)` | `O(1)` |
| 4. Nhị phân + cận chặt + dừng sớm | `O(n log(hi−lo))` | `O(1)` |

In [14]:
rng = random.Random(1760)
big = [rng.randint(1, 10 ** 9) for _ in range(10 ** 5)]
for name, f in [("Cách 2", Solution().minimumSize), ("Cách 3", SolutionBisect().minimumSize), ("Cách 4", SolutionTight().minimumSize)]:
    t = time.perf_counter(); r = f(big, 10 ** 9); dt = time.perf_counter() - t
    print(f"{name}: {r} ({dt*1000:.0f} ms)")

Cách 2: 49915 (109 ms)


Cách 3: 49915 (118 ms)


Cách 4: 49915 (151 ms)


## 12. Bộ test

In [15]:
ALL = {"linear": penalty_linear, "binary": penalty_binary, "submit": Solution().minimumSize,
       "bisect": SolutionBisect().minimumSize, "tight": SolutionTight().minimumSize}
def run_all(nums, ops):
    res = {k: f(nums, ops) for k, f in ALL.items()
           if not (k == "linear" and max(nums) > 10 ** 5)}        # Cách 1' quá chậm với số lớn
    assert len(set(res.values())) == 1, (nums, ops, res)
    return next(iter(res.values()))

for nums, ops, exp in examples:
    assert run_all(nums, ops) == exp
edge = [([1], 1, 1), ([1, 1, 1], 5, 1), ([10], 1, 5), ([10], 9, 1), ([10], 100, 1), ([7, 17], 2, 7), ([10 ** 9], 1, 5 * 10 ** 8)]
for nums, ops, exp in edge:
    assert run_all(nums, ops) == exp, (nums, ops)
rng = random.Random(17600)
for _ in range(250):                                     # đối chiếu vét cạn chia thật (input rất nhỏ)
    nums = [rng.randint(1, 7) for _ in range(rng.randint(1, 3))]
    ops = rng.randint(1, 3)
    assert run_all(nums, ops) == penalty_bruteforce(nums, ops), (nums, ops)
for _ in range(300):                                     # đối chiếu giữa các cách, giá trị vừa
    nums = [rng.randint(1, 2000) for _ in range(rng.randint(1, 30))]
    run_all(nums, rng.randint(1, 200))
print("Ví dụ, edge case, 250 test vét cạn, 300 test ngẫu nhiên: OK")

Ví dụ, edge case, 250 test vét cạn, 300 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: luôn chia đôi túi lớn nhất

In [16]:
def wrong_halve(nums, maxOperations):
    h = [-x for x in nums]; heapq.heapify(h)
    for _ in range(maxOperations):
        x = -heapq.heappop(h)
        if x == 1:
            heapq.heappush(h, -1); break
        heapq.heappush(h, -(x // 2)); heapq.heappush(h, -(x - x // 2))
    return -h[0]

nums, ops, exp = examples[0]
print("sai :", wrong_halve(nums, ops), "| đúng:", exp)

sai : 4 | đúng: 3


**Vì sao sai:** chia đôi `9 → 5 + 4 → (3 + 2) + 4` cho mức phạt 4. Tối ưu là chia **lệch** `9 → 6 + 3 → 3 + 3 + 3`. Quyết định chia thế nào phụ thuộc toàn cục ⇒ dùng nhị phân trên đáp án. **Sửa:** Cách 2.

### Lỗi 2 — Công thức số thao tác `x // p` (quên trừ 1 / quên lấy trần)

In [17]:
def wrong_ops_formula(nums, maxOperations):
    lo, hi = 1, max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(x // mid for x in nums) <= maxOperations:      # SAI
            hi = mid
        else:
            lo = mid + 1
    return lo

for nums, ops, exp in examples:
    print("sai :", wrong_ops_formula(nums, ops), "| đúng:", exp)

sai : 4 | đúng: 3
sai : 3 | đúng: 2


**Vì sao sai:** túi `9` với `p = 3` cần `⌈9/3⌉ − 1 = 2` lần chia, nhưng `9 // 3 = 3` — đếm thừa khi `x` chia hết cho `p`. **Sửa:** `(x − 1) // p`.

### Lỗi 3 — Bắt đầu nhị phân từ `lo = 0`

In [18]:
def wrong_lo0(nums, maxOperations):
    lo, hi = 0, max(nums)
    try:
        while lo < hi:
            mid = (lo + hi) // 2
            if sum((x - 1) // mid for x in nums) <= maxOperations:
                hi = mid
            else:
                lo = mid + 1
        return lo
    except ZeroDivisionError as e:
        return f"ZeroDivisionError: {e}"

print("sai :", wrong_lo0([1], 1), "| đúng:", Solution().minimumSize([1], 1))

sai : ZeroDivisionError: integer division or modulo by zero | đúng: 1


**Vì sao sai:** mức phạt luôn `≥ 1` (túi có số bóng dương); `mid = 0` gây chia cho 0. **Sửa:** `lo = 1`.

### Lỗi 4 — Cập nhật `hi = mid − 1` khi `ok(mid)` (mất đáp án)

In [19]:
def wrong_hi(nums, maxOperations):
    lo, hi = 1, max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((x - 1) // mid for x in nums) <= maxOperations:
            hi = mid - 1                                       # SAI: mid có thể chính là đáp án
        else:
            lo = mid + 1
    return lo

for nums, ops, exp in examples:
    print("sai :", wrong_hi(nums, ops), "| đúng:", exp)

sai : 3 | đúng: 3
sai : 1 | đúng: 2


**Vì sao sai:** với mẫu `while lo < hi` "tìm True đầu tiên", khi `mid` khả thi nó có thể là đáp án nên phải giữ `hi = mid`. **Sửa:** `hi = mid` (hoặc dùng mẫu `while lo <= hi` có biến `ans`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Minimize the maximum**" / "maximize the minimum" ⇒ gần như luôn là **binary search on answer**.
- Viết hàm `need(p)` / `ok(p)` trước, kiểm tra **đơn điệu**, rồi mới viết nhị phân.
- Số lần chia để mọi phần `≤ p`: `(x − 1) // p`.
- Phạm vi tìm: `[1, max]`; cận dưới chặt `⌈Σ/(n+ops)⌉` nếu cần.
- Tham lam cục bộ (chia đôi, chia túi lớn nhất) thường sai cho dạng này — luôn thử phản ví dụ nhỏ.

## 15. Biến thể của bài

1. **875 Koko Eating Bananas**: tốc độ ăn `k`, giờ cần `Σ ⌈x/k⌉ ≤ h` — cùng khuôn.
2. **1011 Capacity To Ship Packages Within D Days**, **410 Split Array Largest Sum**: nhị phân + tham lam kiểm tra.
3. **2064 Minimized Maximum of Products Distributed to Any Store**: `Σ ⌈x/p⌉ ≤ n`.
4. **Trả về cách chia cụ thể** (Bài tập 5).
5. **Mỗi thao tác chia thành 3 túi**: số thao tác cho túi `x` là `⌈(⌈x/p⌉ − 1)/2⌉`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn chia | Thử mọi cách chia | mũ | mũ | Kiểm chứng input tí hon |
| 1'. Thử mọi `p` | `need(p) ≤ ops` | `O(Mn)` | `O(1)` | `M` nhỏ |
| 2. Nhị phân | Đơn điệu của `ok(p)` | `O(n log M)` | `O(1)` | **Bản nộp** |
| 3. `bisect` + key | Như 2, gọn | `O(n log M)` | `O(1)` | Python ≥ 3.10 |
| 4. Cận chặt + dừng sớm | Giảm hằng số | `O(n log M)` | `O(1)` | Muốn nhanh hơn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `([7], 3)`; (b) `([1, 2, 3], 1)`; (c) `([100], 99)`; (d) `([5, 5, 5], 3)`.

**Bài 2 (tính tay):** Với `nums = [7, 17]`, `ops = 2`: lập bảng `need(p)` cho `p = 5..9` và tìm đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `need(nums, p)` trả số thao tác tối thiểu để mọi túi `≤ p`.

**Bài 4 (tự viết lại):** Viết `min_penalty_k_way(nums, ops, k)` khi mỗi thao tác chia một túi thành **k** túi dương (`k ≥ 2`).

**Bài 5 (mini challenge):** Viết `split_plan(nums, ops)` trả về **danh sách túi cuối** đạt mức phạt tối ưu với số thao tác `≤ ops` (kiểm tra: tổng bóng không đổi, số túi `≤ n + ops`, mọi túi dương, max = đáp án).

In [20]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def need_todo(nums, p):
    total = 0
    for x in nums:
        # TODO: cộng số lần chia cho túi x
        pass
    return total

In [21]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def min_penalty_k_way(nums, ops, k):
    # TODO: túi x cần m = ceil(x/p) mảnh; mỗi thao tác thêm (k-1) mảnh -> ceil((m-1)/(k-1)) thao tác
    # TODO: nhớ kiểm tra số mảnh tạo ra không vượt quá x (mỗi mảnh phải dương)
    pass

def split_plan(nums, ops):
    # TODO: tìm p bằng nhị phân, rồi tách mỗi túi thành các túi p (phần dư là một túi nhỏ)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `need(p) = Σ(x−1)//p`. Mức 2: tìm `p` nhỏ nhất `need ≤ ops`. Mức 3: (a) `2`; (b) `2`; (c) `1`; (d) `3`.

**Bài 2.** Mức 1: `need(7) = 0 + 2 = 2`. Mức 2: `need(6) = 1 + 2 = 3 > 2`. Mức 3: đáp án `7`.

**Bài 3.** Mức 1: túi `x` cần `⌈x/p⌉` mảnh. Mức 2: số lần chia = số mảnh − 1. Mức 3: `total += (x − 1) // p`.

**Bài 4.** Mức 1: `m = ⌈x/p⌉`. Mức 2: mỗi thao tác tăng số mảnh thêm `k − 1`. Mức 3: thao tác `t = ⌈(m − 1)/(k − 1)⌉`, tạo ra `1 + t(k − 1)` mảnh — có thể **nhiều hơn** `m`. Mọi mảnh phải dương nên cần `1 + t(k − 1) ≤ x`; nếu không thì mức `p` này **bất khả thi** (ví dụ túi 2 bóng không chia được thành 3 túi). Tính chất này vẫn đơn điệu theo `p`. Kiểm bằng vét cạn nhỏ.

**Bài 5.** Mức 1: `p = Solution().minimumSize(...)`. Mức 2: túi `x` → `(x − 1)//p` túi `p` + một túi `x − p·((x−1)//p)`. Mức 3: kiểm tra các điều kiện nêu trong đề bài tập.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2, 3
b1 = [Solution().minimumSize(n, o) for n, o in [([7], 3), ([1, 2, 3], 1), ([100], 99), ([5, 5, 5], 3)]]
print("Bài 1:", b1); assert b1 == [2, 2, 1, 3]
def need_todo(nums, p):
    total = 0
    for x in nums:
        total += (x - 1) // p
    return total
table = {p: need_todo([7, 17], p) for p in range(5, 10)}
print("Bài 2:", table, "-> đáp án", Solution().minimumSize([7, 17], 2))
assert Solution().minimumSize([7, 17], 2) == 7 and table[6] == 3 and table[7] == 2

Bài 1: [2, 2, 1, 3]
Bài 2: {5: 4, 6: 3, 7: 2, 8: 2, 9: 1} -> đáp án 7


In [23]:
# Bài 4
def min_penalty_k_way(nums, ops, k):
    def need(p):
        total = 0
        for x in nums:
            if x <= p:
                continue
            m = -(-x // p)                       # số mảnh tối thiểu
            t = -(-(m - 1) // (k - 1))           # số thao tác tối thiểu
            if 1 + t * (k - 1) > x:              # không đủ bóng để mọi mảnh đều dương
                return float("inf")
            total += t
        return total
    lo, hi = 1, max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if need(mid) <= ops: hi = mid
        else: lo = mid + 1
    return lo

def brute_k_way(nums, ops, k):
    from itertools import combinations_with_replacement
    @lru_cache(maxsize=None)
    def best(bags, r):
        res = max(bags)
        if r == 0: return res
        for idx, x in enumerate(bags):
            if x >= k:
                rest = bags[:idx] + bags[idx + 1:]
                for parts in combinations_with_replacement(range(1, x), k):
                    if sum(parts) == x:
                        res = min(res, best(tuple(sorted(rest + parts)), r - 1))
        return res
    return best(tuple(sorted(nums)), ops)

assert all(min_penalty_k_way(n, o, 2) == Solution().minimumSize(n, o) for n, o, _ in examples)
rng = random.Random(4)
for _ in range(120):
    nums = [rng.randint(1, 9) for _ in range(rng.randint(1, 2))]; ops = rng.randint(1, 2)
    assert min_penalty_k_way(nums, ops, 3) == brute_k_way(nums, ops, 3), (nums, ops)
print("Bài 4: OK | ví dụ [9], 1 thao tác, chia 3:", min_penalty_k_way([9], 1, 3))

Bài 4: OK | ví dụ [9], 1 thao tác, chia 3: 3


In [24]:
# Bài 5
def split_plan(nums, ops):
    p = Solution().minimumSize(nums, ops)
    bags = []
    for x in nums:
        q = (x - 1) // p
        bags += [p] * q + [x - p * q]
    return p, bags

rng = random.Random(5)
for _ in range(300):
    nums = [rng.randint(1, 50) for _ in range(rng.randint(1, 6))]; ops = rng.randint(1, 20)
    p, bags = split_plan(nums, ops)
    assert sum(bags) == sum(nums) and len(bags) <= len(nums) + ops and min(bags) >= 1 and max(bags) == p
print("Bài 5: OK | ví dụ:", split_plan([2, 4, 8, 2], 4), split_plan([9], 2))

Bài 5: OK | ví dụ: (2, [2, 2, 2, 2, 2, 2, 2, 2]) (3, [3, 3, 3])


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "minimize the maximum":**

```python
lo, hi = 1, max(nums)
while lo < hi:
    mid = (lo + hi) // 2
    if sum((x - 1) // mid for x in nums) <= maxOperations:
        hi = mid
    else:
        lo = mid + 1
return lo
```

**Checklist:**
- [ ] Số thao tác cho túi `x` là `(x − 1) // p` (không phải `x // p`)?
- [ ] `lo = 1` (không chia cho 0), `hi = max(nums)`?
- [ ] Khả thi ⇒ `hi = mid`; không ⇒ `lo = mid + 1`?
- [ ] Không dùng tham lam chia đôi?
- [ ] Đã test túi 1 bóng, `ops` rất lớn, `nums[i] = 10⁹`, `n = 10⁵`?